# Tempo atracado — execução no Google Colab

Roda o experimento completo a partir deste repositório.

1. Na célula abaixo, troque `SEU_USUARIO` pelo seu usuário do GitHub.
2. Envie os `.zip` da ANTAQ (2022, 2023 e 2024) pela barra lateral (ícone de pasta) para `/content`. Opcional: envie também o `resultados.zip` de uma execução anterior para reaproveitar os modelos.
3. Opcional: para reaproveitar os modelos treinados do trabalho em vez de retreinar (cerca de 2 h), cole em `MODELOS_URL` o link do `modelos.zip` da Release v1.0 do repositório.
4. Execute todas as células (*Ambiente de execução → Executar tudo*).

Com o Drive ligado, resultados e modelos ficam em `MyDrive/tcc_tempo_atracado/resultados`: se a sessão cair, basta rodar de novo e o que já foi treinado é reaproveitado.

In [ ]:
REPO_URL = "https://github.com/SEU_USUARIO/tempo-atracado-portos.git"
USAR_DRIVE = True
# Opcional: link do modelos.zip da Release do repositório, para reaproveitar os modelos
# treinados em vez de retreinar (cerca de 2 h). Deixe vazio para treinar do zero.
MODELOS_URL = ""

In [ ]:
import os, sys, glob, shutil

if not os.path.isdir("/content/repo"):
    !git clone -q {REPO_URL} /content/repo
%cd /content/repo
!pip install -q catboost xgboost shap
sys.path.insert(0, "/content/repo/src")

In [ ]:
SAIDA = "/content/resultados"
if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAIDA = "/content/drive/MyDrive/tcc_tempo_atracado/resultados"

# Move os zips enviados para /content até a pasta de dados do repositório
os.makedirs("dados", exist_ok=True)
for z in glob.glob("/content/*.zip"):
    shutil.move(z, os.path.join("dados", os.path.basename(z)))
print(sorted(os.listdir("dados")))

# Baixa os modelos treinados da Release, se MODELOS_URL foi preenchido
if MODELOS_URL:
    import urllib.request, zipfile
    dir_modelos = os.path.join(SAIDA, "modelos")
    os.makedirs(dir_modelos, exist_ok=True)
    print("Baixando os modelos treinados (cerca de 1 GB)...")
    urllib.request.urlretrieve(MODELOS_URL, "/content/modelos_release.zip")
    with zipfile.ZipFile("/content/modelos_release.zip") as zf:
        for nome in zf.namelist():
            base = os.path.basename(nome)
            destino = os.path.join(dir_modelos, base)
            if base.endswith(".joblib") and not os.path.exists(destino):
                with zf.open(nome) as src, open(destino, "wb") as dst:
                    shutil.copyfileobj(src, dst, 1 << 24)
    os.remove("/content/modelos_release.zip")
    print("Modelos disponíveis:", sorted(os.listdir(dir_modelos)))

## Configuração e execução

Os valores padrão reproduzem a versão final do experimento (a do artigo), incluindo os experimentos pedidos no parecer da pré-banca: ablação dos atributos de volume nos quatro modelos e sensibilidade ao corte de extremos (resultados em `resultados/parecer/`). Os experimentos opcionais (XGBoost com categóricas nativas e estabilidade entre sementes) ficam desligados; ligue-os só depois que o essencial rodar.

In [ ]:
from tempo_atracado import Config, executar

cfg = Config(
    dir_dados="dados",
    dir_saida=SAIDA,
    corte_no_teste=False,             # True = comportamento da versão original
    rodar_ablacao=True,               # ablação dos atributos de volume nos quatro modelos
    rodar_sensibilidade_corte=True,   # melhor modelo sem corte e com cortes em 99%, 99,5% e 99,9%
    rodar_xgb_nativo=False,           # opcional, ~10–20 min
    rodar_sementes=False,             # opcional, ~10–30 min
    mostrar_figuras=True,
)
resumo = executar(cfg)

In [ ]:
# Baixa todos os resultados num zip
from google.colab import files
shutil.make_archive("/content/resultados_v3", "zip", SAIDA)
files.download("/content/resultados_v3.zip")